# Task 1: Data acquisition

How to download NHL play-by-play data (2016-17 to 2023-24), cache it locally, and load it back.
All the logic lives in `ift6758/data/nhl.py`; this notebook is the walkthrough you can adapt for the blog post.

**Before running:** copy `.env.example` to `.env` (or `export NHL_DATA_DIR=...`) to choose where raw JSON is cached.
The default is `data/raw/`, which is git-ignored. Never commit the data.

In [11]:
import json
from ift6758.data import (
    get_game, download_season, download_seasons, load_season,
    regular_season_game_id, playoff_game_id,
)

## 1. Game IDs

A game ID is `YYYY` + `TT` + `NNNN`: season start year, game type (`02` regular, `03` playoffs), then a game number.
For playoffs the last four digits are `0` + round + matchup + game number.

In [12]:
print(regular_season_game_id(2016, 1))   # first game of the 2016-17 regular season
print(playoff_game_id(2022, 4, 1, 1))    # 2022-23 Stanley Cup Final, game 1 -> 2022030411

2016020001
2022030411


## 2. Fetch one game

`get_game` checks the cache first. The first call downloads and saves the raw JSON; later calls read from disk without any request.

In [13]:
game = get_game("2022030411")
print(list(game.keys()))
print(len(game["plays"]), "events")
game["plays"][0]

['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor', 'awayTeam', 'homeTeam', 'shootoutInUse', 'otInUse', 'clock', 'displayPeriod', 'gameOutcome', 'plays', 'rosterSpots', 'regPeriods', 'summary']
408 events


{'eventId': 51,
 'periodDescriptor': {'number': 1,
  'periodType': 'REG',
  'maxRegulationPeriods': 3},
 'timeInPeriod': '00:00',
 'timeRemaining': '20:00',
 'situationCode': '1551',
 'homeTeamDefendingSide': 'right',
 'typeCode': 520,
 'typeDescKey': 'period-start',
 'sortOrder': 8}

## 3. Download a whole season

Regular season: counts up from game 1 and stops after a few consecutive 404s (season lengths differ).
Playoffs: walks round, matchup, game 1-7 and moves to the next series at the first missing game.
Re-running is free because everything is cached.

In [14]:
season = download_season(2016)
{k: len(v) for k, v in season.items()}

2016 regular: 1230game [00:05, 219.97game/s] 
2016 playoffs: 87game [00:13,  6.29game/s]


{'regular': 1230, 'playoffs': 87}

## 4. Everything we need: 2016-17 to 2023-24

Slow the first time (be patient, someone else is paying for these requests), instant afterwards.

In [15]:
download_seasons(first=2016, last=2023)


2017 regular: 0game [00:00, ?game/s]

2018 regular: 0game [00:00, ?game/s]


2016 regular: 0game [00:00, ?game/s]
2017 regular: 3game [00:00,  3.06game/s]


2019 regular: 2game [00:00,  2.04game/s]

2016 regular: 3game [00:01,  3.01game/s]
2017 regular: 47game [00:01, 58.59game/s]


2019 regular: 46game [00:01, 57.77game/s]

2016 regular: 58game [00:01, 71.46game/s]
2017 regular: 102game [00:01, 134.03game/s]


2016 regular: 114game [00:01, 145.55game/s]

2018 regular: 111game [00:01, 141.84game/s]
2017 regular: 154game [00:01, 203.22game/s]


2016 regular: 172game [00:01, 222.40game/s]

2018 regular: 165game [00:01, 211.97game/s]
2017 regular: 206game [00:01, 267.64game/s]


2016 regular: 226game [00:01, 286.61game/s]

2018 regular: 218game [00:01, 275.72game/s]
2017 regular: 260game [00:01, 328.64game/s]


2016 regular: 281game [00:01, 344.29game/s]

2018 regular: 272game [00:01, 333.27game/s]
2017 regular: 310game [00:01, 367.38game/s]


2016 regular: 336game [00:01, 394.03game/s]


## 5. Load from disk only

`load_season` never touches the network, which is what you want when building the tidy dataset.

In [16]:
regular = load_season(2016, "regular")
playoffs = load_season(2016, "playoffs")
len(regular), len(playoffs)

(1230, 87)

## 6. Sanity checks

Compare per-season counts against the NHL's published game counts before trusting the data.

In [17]:
for s in range(2016, 2024):
    print(s, len(load_season(s, "regular")), len(load_season(s, "playoffs")))

2016 1230 87
2017 1271 84
2018 1271 87
2019 1082 86
2020 868 84
2021 1312 89
2022 1312 88
2023 1312 88
